# Importación de librerías necesarias

In [6]:
# ═══════════════════════════════════════════════════════════════════════════════
# LIBRERÍAS Y CONFIGURACIÓN DEL ENTORNO
# Objetivo: importar dependencias necesarias para manipulación, análisis,
# visualización y preparación de datos para modelado.
# ═══════════════════════════════════════════════════════════════════════════════

# ──────────────────────────────────────────────────────────────────────────────
# Manipulación y análisis de datos
# ──────────────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ──────────────────────────────────────────────────────────────────────────────
# Visualización de datos
# ──────────────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ──────────────────────────────────────────────────────────────────────────────
# Machine Learning / Preprocesamiento
# ──────────────────────────────────────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

# ──────────────────────────────────────────────────────────────────────────────
# Utilidades
# ──────────────────────────────────────────────────────────────────────────────
import os
import warnings
import requests
from datetime import datetime

warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURACIÓN GLOBAL
# ═══════════════════════════════════════════════════════════════════════════════

# Mostrar todas las columnas del DataFrame
pd.set_option('display.max_columns', None)

# Mostrar números decimales sin notación científica
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

# Importar Archivos

In [7]:
# Fuente: https://www.kaggle.com/datasets/sadiajavedd/social-media-user-activity-dataset/data

# ─────────────────────────────────────────────
# CARGA DE DATOS
# ─────────────────────────────────────────────

# OPCIONES: "nube" (Trabajar directamente con kaggle) | "local" (Si cuentas con el dataset descargado)
FUENTE = "local"

# ── Ruta local (solo se usa si FUENTE = "local") ──────────────
RUTA_LOCAL = "instagram_usage_lifestyle.csv"

# ── Nombre del archivo dentro del dataset de Kaggle ───────────
NOMBRE_ARCHIVO = "instagram_usage_lifestyle.csv"

# ─────────────────────────────────────────────────────────────
if FUENTE == "nube":
    import kagglehub
    print("Descargando dataset desde Kaggle...")
    path = kagglehub.dataset_download("sadiajavedd/social-media-user-activity-dataset")
    ruta = os.path.join(path, NOMBRE_ARCHIVO)
    print(f"Dataset descargado en: {path}")

elif FUENTE == "local":
    ruta = RUTA_LOCAL
    if not os.path.exists(ruta):
        raise FileNotFoundError(
            f"No se encontró el archivo '{ruta}'.\n"
            "Descárgalo manualmente desde:\n"
            "https://www.kaggle.com/datasets/sadiajavedd/social-media-user-activity-dataset/data"
        )
    print(f"Cargando dataset desde archivo local: {ruta}")

else:
    raise ValueError(f"FUENTE inválida: '{FUENTE}'. Usa 'nube' o 'local'.")

# ─────────────────────────────────────────────────────────────

Cargando dataset desde archivo local: instagram_usage_lifestyle.csv


---
# FASE 1 CRISP-DM: Entendimiento del negocio
---

### Contexto del Dataset

Este proyecto utiliza el dataset **Social Media User Activity Dataset** (Kaggle), una colección estructurada de datos sobre el comportamiento de usuarios en redes sociales. Incluye registros de contenido generado por usuarios —publicaciones, captions, hashtags, likes, comentarios, shares y reacciones— junto con métricas de engagement que reflejan la interacción de las audiencias con el contenido.

El dataset captura patrones de comportamiento como frecuencia de publicación, niveles de interacción y popularidad del contenido a lo largo del tiempo, siendo especialmente útil para análisis de tendencias, comportamiento digital y aplicaciones de machine learning.

---

### Definición del Problema

| Elemento | Descripción |
|---|---|
| **Problema** | Predecir el nivel de engagement de usuarios de Instagram |
| **Variable objetivo** | `user_engagement_score` (continua) |
| **Rango observado** | 0.67 – 18.67 |
| **Tipo de tarea** | Regresión supervisada |

---

### Stakeholders

- **Equipos de marketing digital** → Identificar usuarios de alto potencial para campañas segmentadas.
- **Desarrolladores de producto** → Optimizar features que incrementen la participación del usuario.
- **Anunciantes** → Segmentar audiencias con mayor retorno sobre la inversión (ROI).

---

### Criterios de Éxito

| Métrica | Umbral mínimo |
|---|---|
| R² (coeficiente de determinación) | ≥ 0.70 en conjunto de prueba |
| RMSE (raíz del error cuadrático medio) | < 1.5 puntos de engagement |
| MAE (error absoluto medio) | < 1.0 puntos de engagement |

---

### Sobre el Archivo de Datos

El archivo `instagram_usage_lifestyle.csv` contiene datos estructurados de actividad e interacciones en redes sociales con los siguientes atributos principales:

- **Identificadores de usuario** y metadatos de perfil
- **Detalles de publicaciones**: timestamps, captions, hashtags
- **Métricas de engagement**: likes, comentarios, shares, reacciones
- **Features de comportamiento**: frecuencia de publicación, niveles de interacción, popularidad del contenido

Cada registro representa una acción de usuario o ítem de contenido específico, lo que lo hace adecuado para análisis detallados de comportamiento en redes sociales, detección de tendencias y modelado predictivo.

---

In [8]:
print("""
# ═══════════════════════════════════════════════════════════════════════════════
# Exploración inicial de datos
# Objetivo: inspeccionar los primeros 5 registros del DataFrame
# para comprender su estructura, variables y posibles anomalías.
# ═══════════════════════════════════════════════════════════════════════════════
""")
df = pd.read_csv(ruta)

print(f"DataFrame listo — {df.shape[0]:,} filas × {df.shape[1]} columnas")
df.head()


# ═══════════════════════════════════════════════════════════════════════════════
# Exploración inicial de datos
# Objetivo: inspeccionar los primeros 5 registros del DataFrame
# para comprender su estructura, variables y posibles anomalías.
# ═══════════════════════════════════════════════════════════════════════════════

DataFrame listo — 1,547,896 filas × 58 columnas


,user_id,app_name,age,gender,country,urban_rural,income_level,employment_status,education_level,relationship_status,has_children,exercise_hours_per_week,sleep_hours_per_night,diet_quality,smoking,alcohol_frequency,perceived_stress_score,self_reported_happiness,body_mass_index,blood_pressure_systolic,blood_pressure_diastolic,daily_steps_count,weekly_work_hours,hobbies_count,social_events_per_month,books_read_per_year,volunteer_hours_per_month,travel_frequency_per_year,daily_active_minutes_instagram,sessions_per_day,posts_created_per_week,reels_watched_per_day,stories_viewed_per_day,likes_given_per_day,comments_written_per_day,dms_sent_per_week,dms_received_per_week,ads_viewed_per_day,ads_clicked_per_day,time_on_feed_per_day,time_on_explore_per_day,time_on_messages_per_day,time_on_reels_per_day,followers_count,following_count,uses_premium_features,notification_response_rate,account_creation_year,last_login_date,average_session_length_minutes,content_type_preference,preferred_content_theme,privacy_setting_level,two_factor_auth_enabled,biometric_login_used,linked_accounts_count,subscription_status,user_engagement_score
0,1,Instagram,51,Female,India,Rural,High,Retired,Bachelor’s,Single,No,7.20,7.70,Good,No,Rarely,3,8,20.80,148,86,8107,49.90,3,4,7,4.30,0,5.00,1,3,42,28,28,5,12,12,4,1,2,1,1,2,374,647,No,0.34,2015,2025-11-02,5.00,Mixed,Tech,Private,Yes,No,0,Free,7.83
1,2,Instagram,64,Female,United Kingdom,Urban,Middle,Full-time employed,Other,Divorced,No,10.90,8.60,Very poor,No,Rarely,1,1,23.50,133,84,8059,15.60,0,5,10,4.70,2,74.00,5,3,78,54,68,15,18,10,11,1,31,19,16,19,2585,3511,No,0.56,2018,2025-03-22,14.80,Photos,Fashion,Public,No,No,3,Free,1.43
2,3,Instagram,41,Female,Canada,Urban,Middle,Student,Bachelor’s,In a relationship,No,5.00,6.70,Good,No,Rarely,4,10,28.60,135,88,7872,31.80,4,5,14,1.50,2,5.00,1,7,29,26,25,6,12,13,4,0,3,1,1,1,3414,6761,No,0.73,2011,2025-08-10,5.00,Mixed,Other,Public,Yes,Yes,1,Free,9.67
3,4,Instagram,27,Non-binary,South Korea,Urban,Middle,Unemployed,Master’s,In a relationship,No,10.60,6.50,Poor,Yes,Never,18,1,22.50,105,73,7801,43.40,2,3,13,3.30,4,233.00,9,5,241,109,132,36,31,32,33,3,108,64,52,64,617,1193,No,0.73,2019,2025-03-31,25.90,Stories,Tech,Private,No,No,1,Free,0.94
4,5,Instagram,55,Male,India,Urban,Upper-middle,Full-time employed,Bachelor’s,Single,No,7.70,6.80,Average,No,Never,19,1,28.10,146,90,8005,50.20,2,2,12,4.50,3,184.00,14,5,146,113,103,36,29,37,20,5,78,55,22,55,1157,1072,Yes,0.65,2017,2025-03-19,13.10,Videos,Food,Public,Yes,No,0,Free,1.03


In [9]:
print("""
# ═══════════════════════════════════════════════════════════════════════════════
# Información general del DataFrame
# Objetivo: obtener un resumen estructural que incluya número de registros,
# cantidad de columnas, tipos de datos y valores no nulos,
# con el fin de evaluar la calidad y completitud del dataset.
# ═══════════════════════════════════════════════════════════════════════════════
""")

df.info()


# ═══════════════════════════════════════════════════════════════════════════════
# Información general del DataFrame
# Objetivo: obtener un resumen estructural que incluya número de registros,
# cantidad de columnas, tipos de datos y valores no nulos,
# con el fin de evaluar la calidad y completitud del dataset.
# ═══════════════════════════════════════════════════════════════════════════════

<class 'pandas.DataFrame'>
RangeIndex: 1547896 entries, 0 to 1547895
Data columns (total 58 columns):
 #   Column                          Non-Null Count    Dtype  
---  ------                          --------------    -----  
 0   user_id                         1547896 non-null  int64  
 1   app_name                        1547896 non-null  str    
 2   age                             1547896 non-null  int64  
 3   gender                          1547896 non-null  str    
 4   country                         1547896 non-null  str    
 5   urban_rural                     1547896 non-null  str

---
# FASE 2 CRISP-DM: Entendimiento de los datos (EDA)
---

In [10]:
# Nos centramos en nuestra principal variable objetivo

df['score_refinado'] = np.clip(
    0.03 * df['daily_active_minutes_instagram']
    + 0.05 * df['sessions_per_day']
    + 0.01 * (df['reels_watched_per_day'] / 30)
    + 0.008 * df['likes_given_per_day']
    + 0.002 * df['followers_count'] / 1000
    - 0.015 * df['perceived_stress_score']
    + 0.01 * df['self_reported_happiness'],
    0.67, 18.67
).round(2)

df['score_refinado'].describe()

count   1547896.00
mean          6.95
std           3.90
min           0.67
25%           3.86
50%           6.86
75%           9.86
max          18.67
Name: score_refinado, dtype: float64

In [11]:
# Percentiles
p10 = df['score_refinado'].quantile(0.10)
p25 = df['score_refinado'].quantile(0.25)
p75 = df['score_refinado'].quantile(0.75)
p90 = df['score_refinado'].quantile(0.90)

sep = "═" * 50

print(sep)
print("        SEMÁFORO DE ENGAGEMENT SCORE")
print(sep)
print(f"  [CRITICO]    {df['score_refinado'].min():.2f}  <  {p10:.2f}          (P10)")
print(f"  [BAJO]       {p10:.2f}  -  {p25:.2f}          (P10-P25)")
print(f"  [MODERADO]   {p25:.2f}  -  {p75:.2f}          (P25-P75)")
print(f"  [ALTO]       {p75:.2f}  -  {p90:.2f}         (P75-P90)")
print(f"  [MUY ALTO]   {p90:.2f}  >  {df['score_refinado'].max():.2f}        (P90+)")
print(sep)
print(f"  Rango total :  {df['score_refinado'].min():.2f}  ->  {df['score_refinado'].max():.2f}")
print(f"  Media       :  {df['score_refinado'].mean():.2f}")
print(f"  Mediana     :  {df['score_refinado'].median():.2f}")
print(sep)

══════════════════════════════════════════════════
        SEMÁFORO DE ENGAGEMENT SCORE
══════════════════════════════════════════════════
  [CRITICO]    0.67  <  1.51          (P10)
  [BAJO]       1.51  -  3.86          (P10-P25)
  [MODERADO]   3.86  -  9.86          (P25-P75)
  [ALTO]       9.86  -  12.22         (P75-P90)
  [MUY ALTO]   12.22  >  18.67        (P90+)
══════════════════════════════════════════════════
  Rango total :  0.67  ->  18.67
  Media       :  6.95
  Mediana     :  6.86
══════════════════════════════════════════════════
